# 02 — Preprocessing

**Début** : charge `01_feature_engineering` (+ chemin images depuis `00`). **Fin** : manifest + state.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "mlops").is_dir():
        REPO_ROOT = p
        break
sys.path.insert(0, str(REPO_ROOT))

from shared.paths import CARDS_DIR, RAW_CARDS_DIR
from shared.pokemon_dataset import (
    build_tokenizer_and_loaders,
    collect_valid_image_paths,
    pick_device,
    save_preprocessing_manifest,
)
from shared.step_artifacts import (
    load_previous_step_output,
    load_step_output,
    rel_path,
    resolve_path,
    save_step_output,
    step_dir,
)

prev = load_previous_step_output("02_preprocessing")
pull = load_step_output("00_pull_data")

image_dir = resolve_path(pull["cards_dir"])
if not image_dir.exists() or not any(image_dir.rglob("*.png")):
    image_dir = CARDS_DIR if CARDS_DIR.exists() else RAW_CARDS_DIR

print(f"Images: {image_dir}")
print(f"Metadata exemple (étape 01): {prev.get('metadata_path')}")
device = pick_device()


In [ ]:
IMG_SIZE = 512
BATCH_SIZE = 4
VAL_FRACTION = 0.1
SPLIT_SEED = 42

image_paths = collect_valid_image_paths(image_dir)
train_dataset, val_dataset, train_loader, val_loader, tokenizer = build_tokenizer_and_loaders(
    image_dir=image_dir,
    batch_size=BATCH_SIZE,
    img_size=IMG_SIZE,
    val_fraction=VAL_FRACTION,
    split_seed=SPLIT_SEED,
)

manifest_path = step_dir("02_preprocessing") / "preprocessing_manifest.json"
save_preprocessing_manifest(
    manifest_path,
    image_dir=image_dir,
    image_paths=image_paths,
    train_size=len(train_dataset),
    val_size=len(val_dataset),
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    split_seed=SPLIT_SEED,
)

save_step_output("02_preprocessing", {
    "cards_dir": pull["cards_dir"],
    "metadata_path": prev.get("metadata_path"),
    "manifest_path": rel_path(manifest_path),
    "n_images": len(image_paths),
    "train_size": len(train_dataset),
    "val_size": len(val_dataset),
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
})
